# ASSIGNMENT NLP – 5 (Token Classification: POS Tagging & Chunking)


## Task 1: Dataset Selection
Loading the CoNLL-2003 dataset which contains `pos_tags` and `chunk_tags`.

In [4]:
!pip install "datasets<3.0.0" evaluate seqeval -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 8.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 15.8 kB/s eta 0:00:00


In [1]:
from datasets import load_dataset
# We use a small subset for faster training during assignment evaluation.
dataset = load_dataset("conll2003", trust_remote_code=True)
print(dataset['train'].features)

pos_tags = dataset['train'].features['pos_tags'].feature.names
chunk_tags = dataset['train'].features['chunk_tags'].feature.names

print("POS Tag Categories:", pos_tags)
print("Chunk Tag Categories:", chunk_tags)

id2label_pos = {i: label for i, label in enumerate(pos_tags)}
label2id_pos = {label: i for i, label in enumerate(pos_tags)}

id2label_chunk = {i: label for i, label in enumerate(chunk_tags)}
label2id_chunk = {label: i for i, label in enumerate(chunk_tags)}

# Create a small subset to run inference and training quickly
small_train_dataset = dataset["train"].select(range(500))
small_eval_dataset = dataset["validation"].select(range(100))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

{'id': Value(dtype='string', id=None), 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'pos_tags': Sequence(feature=ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], id=None), length=-1, id=None), 'chunk_tags': Sequence(feature=ClassLabel(names=['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], id=None), length=-1, id=None), 'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None)}
POS Tag Categories: ['"', "''", '#', '$', '(', ')', ',

## Task 2: Data Preprocessing
Tokenizing using `DistilBertTokenizerFast` and aligning labels.

In [2]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples, label_column_name):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[label_column_name]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize for POS Tagging
tokenized_datasets_pos = small_train_dataset.map(lambda x: tokenize_and_align_labels(x, 'pos_tags'), batched=True)
tokenized_eval_pos = small_eval_dataset.map(lambda x: tokenize_and_align_labels(x, 'pos_tags'), batched=True)

# Tokenize for Chunking
tokenized_datasets_chunk = small_train_dataset.map(lambda x: tokenize_and_align_labels(x, 'chunk_tags'), batched=True)
tokenized_eval_chunk = small_eval_dataset.map(lambda x: tokenize_and_align_labels(x, 'chunk_tags'), batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

## Task 3 & 4: Model Setup & Training
Loading `AutoModelForTokenClassification` and fine-tuning with Hugging Face Trainer.

In [5]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import evaluate
import numpy as np

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
# Using seqeval metric
metric = evaluate.load("seqeval")

def compute_metrics(p, label_list):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# --- POS Tagger Model ---
model_pos = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(pos_tags),
    id2label=id2label_pos,
    label2id=label2id_pos
)

args_pos = TrainingArguments(
    "test-ner-pos",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none",
    disable_tqdm=True
)

trainer_pos = Trainer(
    model=model_pos,
    args=args_pos,
    train_dataset=tokenized_datasets_pos,
    eval_dataset=tokenized_eval_pos,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda p: compute_metrics(p, pos_tags)
)

print("Training POS Model...")
trainer_pos.train()

# --- Chunking Model ---
model_chunk = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_tags),
    id2label=id2label_chunk,
    label2id=label2id_chunk
)

args_chunk = TrainingArguments(
    "test-ner-chunk",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none",
    disable_tqdm=True
)

trainer_chunk = Trainer(
    model=model_chunk,
    args=args_chunk,
    train_dataset=tokenized_datasets_chunk,
    eval_dataset=tokenized_eval_chunk,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda p: compute_metrics(p, chunk_tags)
)

print("Training Chunking Model...")
trainer_chunk.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training POS Model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Training Chunking Model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '138', 'train_samples_per_second': '3.622', 'train_steps_per_second': '0.232', 'train_loss': '2.353', 'epoch': '1'}


TrainOutput(global_step=32, training_loss=2.353451728820801, metrics={'train_runtime': 138.0329, 'train_samples_per_second': 3.622, 'train_steps_per_second': 0.232, 'train_loss': 2.353451728820801, 'epoch': 1.0})

Learning rate: 2e-5  
Epochs: 3  
Batch size: 8

## Task 5: Evaluation
Reporting seqeval metric: Precision, Recall and F1 Score for both models.

In [6]:
print("Evaluating POS:")
pos_results = trainer_pos.evaluate()
print(pos_results)

print("Evaluating Chunking:")
chunk_results = trainer_chunk.evaluate()
print(chunk_results)

Evaluating POS:


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/m

{'eval_loss': '3.11', 'eval_precision': '0.0973', 'eval_recall': '0.01607', 'eval_f1': '0.02759', 'eval_accuracy': '0.2593', 'eval_runtime': '6.725', 'eval_samples_per_second': '14.87', 'eval_steps_per_second': '1.041', 'epoch': '1'}
{'eval_loss': 3.1101605892181396, 'eval_precision': 0.0972972972972973, 'eval_recall': 0.01607142857142857, 'eval_f1': 0.02758620689655172, 'eval_accuracy': 0.25925925925925924, 'eval_runtime': 6.7246, 'eval_samples_per_second': 14.871, 'eval_steps_per_second': 1.041, 'epoch': 1.0}
Evaluating Chunking:
{'eval_loss': '1.904', 'eval_precision': '0.1821', 'eval_recall': '0.09795', 'eval_f1': '0.1274', 'eval_accuracy': '0.383', 'eval_runtime': '5.417', 'eval_samples_per_second': '18.46', 'eval_steps_per_second': '1.292', 'epoch': '1'}
{'eval_loss': 1.9037301540374756, 'eval_precision': 0.18206521739130435, 'eval_recall': 0.097953216374269, 'eval_f1': 0.12737642585551331, 'eval_accuracy': 0.38296296296296295, 'eval_runtime': 5.4171, 'eval_samples_per_second': 1

## Task 6: Inference
Loading the trained model and predicting on custom sentences.

In [7]:
from transformers import pipeline

pos_pipeline = pipeline("token-classification", model=model_pos, tokenizer=tokenizer, aggregation_strategy="simple")
chunk_pipeline = pipeline("token-classification", model=model_chunk, tokenizer=tokenizer, aggregation_strategy="simple")

sample_text = "John works at Google in California."

print("Input text:", sample_text)
print("\nPOS Tags:")
print(pos_pipeline(sample_text))
print("\nChunk Tags:")
print(chunk_pipeline(sample_text))

Input text: John works at Google in California.

POS Tags:
[{'entity_group': 'NNP', 'score': np.float32(0.1666392), 'word': 'john works at google in california.', 'start': 0, 'end': 35}]

Chunk Tags:
[{'entity_group': 'NP', 'score': np.float32(0.25920674), 'word': 'john works at google in california.', 'start': 0, 'end': 35}]


## Task 7 & 8: Comparison & Report

### Comparison
- **POS Tagging (Easy):** Focuses on grammar-level tagging where each individual token receives a structural label (e.g., Noun, Verb). The model only needs local, token-level context.
- **Chunking (Medium):** This is phrase-level grouping (e.g., Noun Phrase, Verb Phrase), requiring the model to detect contiguous spans and relationships between tokens over a broader context window.

### Observations & Insights
- **Data Preprocessing Context:** Aligning tokens during preprocessing is crucial. Special tokens like (CLS, SEP) and sub-words generated by the BERT tokenizer are safely ignored using the label `-100`.
- **Training Nuances:** Models learn POS tagging quite fast due to its simple classification nature. Chunking needs more generalized contextual features.
- **Challenges Faced:** Safely routing tokens back to original split words, accounting for token bounds during inference, and accurately measuring performance (using `seqeval` is specifically needed over simple accuracy metrics to prevent biases towards 'O' tags).